In [ ]:
# Invisibility Cloak App using Gradio (Notebook-compatible)

import gradio as gr
import os
import cv2
import numpy as np
import urllib.request
import zipfile
from tempfile import NamedTemporaryFile

# Download YOLO weights and config if not present
def download_yolo_files():
    if not os.path.exists("yolov3.weights"):
        print("Downloading yolov3.weights...")
        urllib.request.urlretrieve(
            "https://pjreddie.com/media/files/yolov3.weights", "yolov3.weights")
    if not os.path.exists("yolov3.cfg"):
        print("Downloading yolov3.cfg...")
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/pjreddie/darknet/master/cfg/yolov3.cfg", "yolov3.cfg")
    if not os.path.exists("coco.names"):
        print("Downloading coco.names...")
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/pjreddie/darknet/master/data/coco.names", "coco.names")

# Call download on startup
download_yolo_files()

# Invisibility Cloak Processing Function (returns processed video path)
def invisibility_cloak(video_path):
    # Load YOLO model
    net = cv2.dnn.readNet("yolov3.weights", "yolov3.cfg")
    net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
    net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)

    # Load class names
    with open("coco.names", "r") as f:
        classes = [line.strip() for line in f.readlines()]
    layer_names = net.getLayerNames()
    output_layers = [layer_names[i - 1] for i in net.getUnconnectedOutLayers().flatten()]

    # Open uploaded video
    cap = cv2.VideoCapture(video_path)
    ret, background = cap.read()
    if not ret:
        raise Exception("Error: Cannot read video or first frame.")
    
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    output_temp = NamedTemporaryFile(delete=False, suffix=".mp4")
    out = cv2.VideoWriter(output_temp.name, fourcc, fps, (frame_width, frame_height))

    background_gray = cv2.cvtColor(background, cv2.COLOR_BGR2GRAY)
    lower_whitegrey = np.array([0, 0, 130])
    upper_whitegrey = np.array([180, 50, 255])

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Optical Flow Alignment
        flow = cv2.calcOpticalFlowFarneback(background_gray, frame_gray, None,
                                            0.5, 5, 25, 5, 7, 1.5, 0)
        y, x = np.mgrid[0:frame_height, 0:frame_width].astype(np.float32)
        remap_x = x + flow[..., 0]
        remap_y = y + flow[..., 1]
        aligned_bg = cv2.remap(background, remap_x, remap_y, cv2.INTER_LINEAR)

        # YOLO Detection
        blob = cv2.dnn.blobFromImage(frame, 0.00392, (416, 416),
                                     (0, 0, 0), True, crop=False)
        net.setInput(blob)
        outputs = net.forward(output_layers)

        boxes = []
        confidences = []

        for output in outputs:
            for detection in output:
                scores = detection[5:]
                class_id = np.argmax(scores)
                confidence = scores[class_id]
                if class_id == 0 and confidence > 0.5:
                    center_x = int(detection[0] * frame_width)
                    center_y = int(detection[1] * frame_height)
                    w = int(detection[2] * frame_width)
                    h = int(detection[3] * frame_height)
                    x = int(center_x - w / 2)
                    y = int(center_y - h / 2)
                    boxes.append([x, y, w, h])
                    confidences.append(float(confidence))

        indices = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.4)

        if len(indices) > 0:
            for i in indices.flatten():
                x, y, w, h = boxes[i]
                x = max(0, x)
                y = max(0, y)
                w = min(w, frame_width - x)
                h = min(h, frame_height - y)
                roi = frame[y:y + h, x:x + w]
                if roi.size == 0:
                    continue

                hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
                mask = cv2.inRange(hsv, lower_whitegrey, upper_whitegrey)
                ratio = cv2.countNonZero(mask) / (roi.shape[0] * roi.shape[1])

                if ratio > 0.5:
                    frame[y:y + h, x:x + w] = aligned_bg[y:y + h, x:x + w]

        out.write(frame)

    cap.release()
    out.release()
    return output_temp.name

# Gradio Interface
interface = gr.Interface(
    fn=invisibility_cloak,
    inputs=gr.Video(label="Upload a Video"),
    outputs=gr.Video(label="Cloaked Output"),
    title="Invisibility Cloak using YOLO + Optical Flow",
    description="Detects humans wearing white/grey clothes and cloaks them using background reconstruction."
)

# Launch
interface.launch(share=True)
